In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime


# Загружаем данные
current_dir = Path.cwd()
data_path = current_dir / "data" / "raw" / "ml-25m"
if not data_path.exists():
    data_path = current_dir.parent / "data" / "raw" / "ml-25m"

ratings = pd.read_csv(data_path / "ratings.csv")
movies = pd.read_csv(data_path / "movies.csv")

ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['date'] = ratings['datetime'].dt.date
user_activity = ratings.merge(movies, on='movieId', how='left')

# Рассчитываем метрики
dau_daily = user_activity.groupby('date')['userId'].nunique()
avg_dau = dau_daily.mean()
mau = user_activity[user_activity['date'] >= (user_activity['date'].max() - pd.Timedelta(days=30))]['userId'].nunique()
stickiness = avg_dau / mau if mau > 0 else 0

avg_rating = ratings['rating'].mean()
high_rating_ratio = (ratings['rating'] >= 4.0).mean()


report_lines = []
report_lines.append("=" * 70)
report_lines.append("ОТЧЕТ: MovieLens Analysis")
report_lines.append("=" * 70)
report_lines.append(f"Дата подготовки: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
report_lines.append("")

# Раздел 1
report_lines.append("=" * 70)
report_lines.append("РАЗДЕЛ 1: ОБЩИЕ ПОКАЗАТЕЛИ")
report_lines.append("=" * 70)
report_lines.append(f"Всего оценок:          {len(ratings):>20,}")
report_lines.append(f"Всего фильмов:         {movies.shape[0]:>20,}")
report_lines.append(f"Всего пользователей:   {ratings['userId'].nunique():>20,}")
report_lines.append(f"Период данных:         {ratings['date'].min()} - {ratings['date'].max()}")
report_lines.append("")

# Раздел 2
report_lines.append("=" * 70)
report_lines.append("РАЗДЕЛ 2: КЛЮЧЕВЫЕ МЕТРИКИ")
report_lines.append("=" * 70)
report_lines.append("")
report_lines.append("ENGAGEMENT:")
report_lines.append(f"  • Avg DAU:            {avg_dau:>20,.0f}")
report_lines.append(f"  • MAU (30d):          {mau:>20,}")
report_lines.append(f"  • Stickiness:         {stickiness:>20.2%}")
report_lines.append("")
report_lines.append("QUALITY:")
report_lines.append(f"  • Avg Rating:         {avg_rating:>20.2f}")
report_lines.append(f"  • High Rating (≥4.0): {high_rating_ratio:>20.1%}")
report_lines.append("")

# Раздел 3
report_lines.append("=" * 70)
report_lines.append("РАЗДЕЛ 3: ИНСАЙТЫ")
report_lines.append("=" * 70)
report_lines.append("")

if stickiness > 0.15:
    report_lines.append(" Высокая вовлеченность пользователей (Stickiness)")
if avg_rating >= 3.5:
    report_lines.append(" Хорошее качество контента (Avg Rating ≥ 3.5)")
if high_rating_ratio > 0.6:
    report_lines.append(" Большинство оценок положительные (>60%)")
if stickiness <= 0.15:
    report_lines.append(" Низкая частота возвращений (Stickiness < 15%)")
if avg_rating < 3.5:
    report_lines.append(" Средняя оценка ниже ожидаемой")

report_lines.append("")

# Раздел 4
report_lines.append("=" * 70)
report_lines.append("РАЗДЕЛ 4: РЕКОМЕНДАЦИИ")
report_lines.append("=" * 70)
report_lines.append("")
report_lines.append("1. Retention:")
report_lines.append("   • Внедрить push-уведомления о новых фильмах")
report_lines.append("   • Создать персонализированные рекомендации")
report_lines.append("   • Запустить программу лояльности")
report_lines.append("")
report_lines.append("2. Quality:")
report_lines.append("   • Улучшить алгоритм подбора контента")
report_lines.append("   • Добавить больше популярных жанров")
report_lines.append("   • Стимулировать пользователей оценивать фильмы")
report_lines.append("")
report_lines.append("3. Engagement:")
report_lines.append("   • Добавить социальные функции (списки, рецензии)")
report_lines.append("   • Создать челленджи и достижения")
report_lines.append("   • Улучшить UX/UI платформы")
report_lines.append("")

# Раздел 5
report_lines.append("=" * 70)
report_lines.append("РАЗДЕЛ 5: ВЫВОДЫ")
report_lines.append("=" * 70)
report_lines.append("")

overall_score = " ОТЛИЧНО" if (stickiness > 0.15 and avg_rating >= 3.5) else " ХОРОШО"
report_lines.append(f"ОБЩАЯ ОЦЕНКА ПРОДУКТА: {overall_score}")
report_lines.append("")
report_lines.append(f"MovieLens показывает {'хорошие' if stickiness > 0.1 else 'удовлетворительные'} результаты.")
report_lines.append("Платформа имеет активную базу пользователей и качественный контент.")
report_lines.append("")
report_lines.append("Основные направления развития:")
report_lines.append("  • Увеличение частоты возвращений пользователей")
report_lines.append("  • Расширение контентной библиотеки")
report_lines.append("  • Улучшение персонализации")
report_lines.append("")
report_lines.append("=" * 70)

# ============================================
# СОХРАНЯЕМ В ФАЙЛ
# ============================================
docs_path = current_dir / "docs"
if not docs_path.exists():
    docs_path = current_dir.parent / "docs"

report_file = docs_path / "final_report.txt"
with open(report_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

print(f"Отчет сохранен: {report_file}")
print("")


from IPython.display import display, Markdown, HTML

# Показываем первые 30 строк
preview = '\n'.join(report_lines[:40])
display(Markdown(f"```\n{preview}\n```\n\n**Полный отчет сохранен в:** `{report_file}`"))

# Показываем ключевые метрики отдельно
metrics_html = f"""
<div style='background-color: #f5f5f5; padding: 20px; border-radius: 10px; font-family: monospace;'>
<h3> Ключевые метрики</h3>
<table style='width: 100%;'>
<tr><td>Stickiness:</td><td><b>{stickiness:.2%}</b></td></tr>
<tr><td>Avg Rating:</td><td><b>{avg_rating:.2f}</b></td></tr>
<tr><td>High Rating Ratio:</td><td><b>{high_rating_ratio:.1%}</b></td></tr>
<tr><td>Overall Score:</td><td><b>{overall_score}</b></td></tr>
</table>
</div>
"""

display(HTML(metrics_html))

Отчет сохранен: c:\Users\Мартинсон Диана\metric tree\docs\final_report.txt



```
======================================================================
ОТЧЕТ: MovieLens Analysis
======================================================================
Дата подготовки: 2026-04-17 12:09

======================================================================
РАЗДЕЛ 1: ОБЩИЕ ПОКАЗАТЕЛИ
======================================================================
Всего оценок:                    25,000,095
Всего фильмов:                       62,423
Всего пользователей:                162,541
Период данных:         1995-01-09 - 2019-11-21

======================================================================
РАЗДЕЛ 2: КЛЮЧЕВЫЕ МЕТРИКИ
======================================================================

ENGAGEMENT:
  • Avg DAU:                             162
  • MAU (30d):                         2,431
  • Stickiness:                        6.65%

QUALITY:
  • Avg Rating:                         3.53
  • High Rating (≥4.0):                49.8%

======================================================================
РАЗДЕЛ 3: ИНСАЙТЫ
======================================================================

 Хорошее качество контента (Avg Rating ≥ 3.5)
 Низкая частота возвращений (Stickiness < 15%)

======================================================================
РАЗДЕЛ 4: РЕКОМЕНДАЦИИ
======================================================================

1. Retention:
   • Внедрить push-уведомления о новых фильмах
   • Создать персонализированные рекомендации
```

**Полный отчет сохранен в:** `c:\Users\Мартинсон Диана\metric tree\docs\final_report.txt`

Stickiness:,6.65%
Avg Rating:,3.53
High Rating Ratio:,49.8%
Overall Score:,ХОРОШО


In [6]:
# ============================================
# ФИНАЛЬНЫЙ ОТЧЕТ — ПОЛНЫЙ ВЫВОД
# ============================================
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown

# Загружаем данные
current_dir = Path.cwd()
data_path = current_dir / "data" / "raw" / "ml-25m"
if not data_path.exists():
    data_path = current_dir.parent / "data" / "raw" / "ml-25m"

ratings = pd.read_csv(data_path / "ratings.csv")
movies = pd.read_csv(data_path / "movies.csv")

ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['date'] = ratings['datetime'].dt.date
user_activity = ratings.merge(movies, on='movieId', how='left')

# Рассчитываем метрики
dau_daily = user_activity.groupby('date')['userId'].nunique()
avg_dau = dau_daily.mean()
mau = user_activity[user_activity['date'] >= (user_activity['date'].max() - pd.Timedelta(days=30))]['userId'].nunique()
stickiness = avg_dau / mau if mau > 0 else 0

avg_rating = ratings['rating'].mean()
high_rating_ratio = (ratings['rating'] >= 4.0).mean()

# Сохраняем в файл
docs_path = current_dir / "docs"
if not docs_path.exists():
    docs_path = current_dir.parent / "docs"

report_file = docs_path / "final_report.txt"

with open(report_file, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("ОТЧЕТ: MovieLens Analysis\n")
    f.write("=" * 70 + "\n")
    f.write(f"Дата подготовки: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    
    f.write("=" * 70 + "\n")
    f.write("РАЗДЕЛ 1: ОБЩИЕ ПОКАЗАТЕЛИ\n")
    f.write("=" * 70 + "\n")
    f.write(f"Всего оценок:          {len(ratings):>20,}\n")
    f.write(f"Всего фильмов:         {movies.shape[0]:>20,}\n")
    f.write(f"Всего пользователей:   {ratings['userId'].nunique():>20,}\n")
    f.write(f"Период данных:         {ratings['date'].min()} - {ratings['date'].max()}\n\n")
    
    f.write("=" * 70 + "\n")
    f.write("РАЗДЕЛ 2: КЛЮЧЕВЫЕ МЕТРИКИ\n")
    f.write("=" * 70 + "\n\n")
    f.write("ENGAGEMENT:\n")
    f.write(f"  Avg DAU:                 {avg_dau:>15,.0f}\n")
    f.write(f"  MAU (30d):               {mau:>15,}\n")
    f.write(f"  Stickiness:              {stickiness:>15.2%}\n\n")
    f.write("QUALITY:\n")
    f.write(f"  Avg Rating:              {avg_rating:>15.2f}\n")
    f.write(f"  High Rating (≥4.0):      {high_rating_ratio:>15.1%}\n\n")
    
    f.write("=" * 70 + "\n")
    f.write("РАЗДЕЛ 3: ИНСАЙТЫ\n")
    f.write("=" * 70 + "\n\n")
    
    if stickiness > 0.15:
        f.write("✓ Высокая вовлеченность пользователей (Stickiness)\n")
    if avg_rating >= 3.5:
        f.write("✓ Хорошее качество контента (Avg Rating ≥ 3.5)\n")
    if high_rating_ratio > 0.6:
        f.write("✓ Большинство оценок положительные (>60%)\n")
    if stickiness <= 0.15:
        f.write("! Низкая частота возвращений (Stickiness < 15%)\n")
    if avg_rating < 3.5:
        f.write("! Средняя оценка ниже ожидаемой\n")
    
    f.write("\n" + "=" * 70 + "\n")
    f.write("РАЗДЕЛ 4: РЕКОМЕНДАЦИИ\n")
    f.write("=" * 70 + "\n\n")
    f.write("1. Retention:\n")
    f.write("   - Внедрить push-уведомления о новых фильмах\n")
    f.write("   - Создать персонализированные рекомендации\n")
    f.write("   - Запустить программу лояльности\n\n")
    f.write("2. Quality:\n")
    f.write("   - Улучшить алгоритм подбора контента\n")
    f.write("   - Добавить больше популярных жанров\n")
    f.write("   - Стимулировать пользователей оценивать фильмы\n\n")
    f.write("3. Engagement:\n")
    f.write("   - Добавить социальные функции (списки, рецензии)\n")
    f.write("   - Создать челленджи и достижения\n")
    f.write("   - Улучшить UX/UI платформы\n\n")
    
    f.write("=" * 70 + "\n")
    f.write("РАЗДЕЛ 5: ВЫВОДЫ\n")
    f.write("=" * 70 + "\n\n")
    
    overall_score = "ОТЛИЧНО" if (stickiness > 0.15 and avg_rating >= 3.5) else "ХОРОШО"
    f.write(f"ОБЩАЯ ОЦЕНКА ПРОДУКТА: {overall_score}\n\n")
    f.write(f"MovieLens показывает {'хорошие' if stickiness > 0.1 else 'удовлетворительные'} результаты.\n")
    f.write("Платформа имеет активную базу пользователей и качественный контент.\n\n")
    f.write("Основные направления развития:\n")
    f.write("  - Увеличение частоты возвращений пользователей\n")
    f.write("  - Расширение контентной библиотеки\n")
    f.write("  - Улучшение персонализации\n\n")
    f.write("=" * 70 + "\n")

# Показываем отчет ВЕСЬ полностью через Markdown
with open(report_file, 'r', encoding='utf-8') as f:
    full_report = f.read()

# Показываем как код (без обрезки)
display(Markdown(f"```\n{full_report}\n```"))

print(f"\n✅ Полный отчет сохранен: {report_file}")

```
======================================================================
ОТЧЕТ: MovieLens Analysis
======================================================================
Дата подготовки: 2026-04-17 12:48

======================================================================
РАЗДЕЛ 1: ОБЩИЕ ПОКАЗАТЕЛИ
======================================================================
Всего оценок:                    25,000,095
Всего фильмов:                       62,423
Всего пользователей:                162,541
Период данных:         1995-01-09 - 2019-11-21

======================================================================
РАЗДЕЛ 2: КЛЮЧЕВЫЕ МЕТРИКИ
======================================================================

ENGAGEMENT:
  Avg DAU:                             162
  MAU (30d):                         2,431
  Stickiness:                        6.65%

QUALITY:
  Avg Rating:                         3.53
  High Rating (≥4.0):                49.8%

======================================================================
РАЗДЕЛ 3: ИНСАЙТЫ
======================================================================

✓ Хорошее качество контента (Avg Rating ≥ 3.5)
! Низкая частота возвращений (Stickiness < 15%)

======================================================================
РАЗДЕЛ 4: РЕКОМЕНДАЦИИ
======================================================================

1. Retention:
   - Внедрить push-уведомления о новых фильмах
   - Создать персонализированные рекомендации
   - Запустить программу лояльности

2. Quality:
   - Улучшить алгоритм подбора контента
   - Добавить больше популярных жанров
   - Стимулировать пользователей оценивать фильмы

3. Engagement:
   - Добавить социальные функции (списки, рецензии)
   - Создать челленджи и достижения
   - Улучшить UX/UI платформы

======================================================================
РАЗДЕЛ 5: ВЫВОДЫ
======================================================================

ОБЩАЯ ОЦЕНКА ПРОДУКТА: ХОРОШО

MovieLens показывает удовлетворительные результаты.
Платформа имеет активную базу пользователей и качественный контент.

Основные направления развития:
  - Увеличение частоты возвращений пользователей
  - Расширение контентной библиотеки
  - Улучшение персонализации

======================================================================

```


✅ Полный отчет сохранен: c:\Users\Мартинсон Диана\metric tree\docs\final_report.txt
